# Task 1: Feature Engineering Pipeline
**Objective:** Transform raw transactional and temporal data into behavioral signals that differentiate human behavior from automated fraud bot activity.
**Key Features to be Developed:**
1. **time_since_signup:** Duration between account creation and purchase.
2. **Transaction Velocity:** Frequency of device usage across different users.
3. **Temporal Patterns:** Hour of day and day of week trends.

In [ ]:
# Setup & Data Ingestion 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

# Path setup to access modular src code
sys.path.append(os.path.abspath('../'))
from src.preprocessing import merge_ip_to_country, FraudPreprocessor

# Load Raw Data
fraud_df = pd.read_csv('../data/raw/Fraud_Data.csv')
ip_df = pd.read_csv('../data/raw/IpAddress_to_Country.csv')

# Step 1: Geolocation Integration
merged_df = merge_ip_to_country(fraud_df, ip_df)
print(f"Data Ingested. Shape: {merged_df.shape}")

In [ ]:
# Feature Engineering (Behavioral)
# Convert to datetime
merged_df['signup_time'] = pd.to_datetime(merged_df['signup_time'])
merged_df['purchase_time'] = pd.to_datetime(merged_df['purchase_time'])

# 1. time_since_signup (Hours)
# Hypothesis: Fraudsters buy immediately after signup to maximize bot efficiency.
merged_df['time_since_signup'] = (merged_df['purchase_time'] - merged_df['signup_time']).dt.total_seconds() / 3600

# 2. Transaction Velocity (user_per_device)
# Hypothesis: High user-to-device ratios indicate 'Fraud Farms'.
merged_df['user_per_device'] = merged_df.groupby('device_id')['user_id'].transform('count')

# 3. IP Velocity
merged_df['user_per_ip'] = merged_df.groupby('ip_address')['user_id'].transform('count')

print("Behavioral signals engineered.")

In [ ]:
# Feature Engineering (Temporal)
# Extract temporal components
merged_df['hour_of_day'] = merged_df['purchase_time'].dt.hour
merged_df['day_of_week'] = merged_df['purchase_time'].dt.dayofweek

print("Temporal signals engineered.")